In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Directorio actual:", os.getcwd())

Directorio actual: /Users/sofiajimenez/Documents/5to semestre/MineriaDatosT/mibici/DatosPoblacion


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")
%matplotlib inline

In [3]:
import os
print("Directorio actual:", os.getcwd())
print("\nContenido de data/raw/:")
for f in sorted(os.listdir('data/raw')):
    print(" -", repr(f))

Directorio actual: /Users/sofiajimenez/Documents/5to semestre/MineriaDatosT/mibici/DatosPoblacion

Contenido de data/raw/:
 - '.gitignore'
 - 'conjunto_de_datos_hog_enti_2022.csv'
 - 'conjunto_de_datos_hog_enti_2022.csv.dvc'
 - 'conjunto_de_datos_viv_enti_2022.csv'
 - 'conjunto_de_datos_viv_enti_2022.csv.dvc'
 - 'diccionario_datos_hog_enti_2022.csv'
 - 'diccionario_datos_viv_enti_2022.csv'


In [4]:
# Diccionario Hogar
dict_hog = pd.read_csv('data/raw/diccionario_datos_hog_enti_2022.csv', encoding='utf-8')
dict_hog.columns = ['NOMBRE_CAMPO', 'LONGITUD', 'TIPO', 'NEMONICO', 'CATALOGO', 'RANGO_CLAVES']

# Diccionario Vivienda
dict_viv = pd.read_csv('data/raw/diccionario_datos_viv_enti_2022.csv', encoding='utf-8')
dict_viv.columns = ['NOMBRE_CAMPO', 'LONGITUD', 'TIPO', 'NEMONICO', 'CATALOGO', 'RANGO_CLAVES']

print("Diccionario Hogar:", dict_hog.shape)
print("Diccionario Vivienda:", dict_viv.shape)
print("\nColumnas estandarizadas:", dict_hog.columns.tolist())

Diccionario Hogar: (132, 6)
Diccionario Vivienda: (54, 6)

Columnas estandarizadas: ['NOMBRE_CAMPO', 'LONGITUD', 'TIPO', 'NEMONICO', 'CATALOGO', 'RANGO_CLAVES']


In [5]:
df_hog = pd.read_csv('data/raw/conjunto_de_datos_hog_enti_2022.csv', dtype=str, encoding='utf-8')
df_viv = pd.read_csv('data/raw/conjunto_de_datos_viv_enti_2022.csv', dtype=str, encoding='utf-8')

print("Hogar:", df_hog.shape)
print("Vivienda:", df_viv.shape)
print("\nPrimeras columnas de Hogar:", df_hog.columns.tolist()[:10])
print("\nPrimeras columnas de Vivienda:", df_viv.columns.tolist()[:10])

Hogar: (65862, 132)
Vivienda: (64803, 54)

Primeras columnas de Hogar: ['est', 'est_d', 't_loc', 'ent', 'con', 'upm', 'n_pro_viv', 'v_sel', 'n_hog', 'per']

Primeras columnas de Vivienda: ['est', 'est_d', 't_loc', 'ent', 'con', 'upm', 'n_pro_viv', 'v_sel', 'per', 'p1']


In [6]:
llaves = ['ent', 'con', 'upm', 'n_pro_viv', 'v_sel', 'per']

print("¿Hogar tiene todas las llaves?")
for l in llaves:
    print(f"  {l}: {l in df_hog.columns}")

print("\n¿Vivienda tiene todas las llaves?")
for l in llaves:
    print(f"  {l}: {l in df_viv.columns}")



¿Hogar tiene todas las llaves?
  ent: True
  con: True
  upm: True
  n_pro_viv: True
  v_sel: True
  per: True

¿Vivienda tiene todas las llaves?
  ent: True
  con: True
  upm: True
  n_pro_viv: True
  v_sel: True
  per: True


In [7]:
print("Hogar:", df_hog.shape)
print("Vivienda:", df_viv.shape)
print("Diccionario Hogar:", dict_hog.shape)
print("Diccionario Vivienda:", dict_viv.shape)

Hogar: (65862, 132)
Vivienda: (64803, 54)
Diccionario Hogar: (132, 6)
Diccionario Vivienda: (54, 6)


In [8]:
llaves = ['ent', 'con', 'upm', 'n_pro_viv', 'v_sel', 'per']

df_integrado = pd.merge(
    df_hog,
    df_viv,
    on=llaves,
    how='inner',
    suffixes=('_hog', '_viv')
)

print("Hogar original:", df_hog.shape)
print("Vivienda original:", df_viv.shape)
print("Integrado:", df_integrado.shape)
print("Columnas esperadas: 132 + 54 - 6 =", 132 + 54 - 6)

Hogar original: (65862, 132)
Vivienda original: (64803, 54)
Integrado: (65862, 180)
Columnas esperadas: 132 + 54 - 6 = 180


In [9]:
#¿Qué relación existe entre el trabajo infantil en el hogar y las condiciones de la vivienda y la escolaridad del jefe?

In [10]:
columnas_relevantes = [
    # Llaves (compartidas)
    'ent', 'con', 'upm', 'n_pro_viv', 'v_sel', 'per',
    # De Hogar
    'tcti', 'tonp', 'tocup', 'tqdna', 'con5a17',
    'escol_jefe', 'cs_p13_2', 'sex_jef', 'clase_hog',
    'tam_hog', 'p_tot', 'clase2_jef',
    # De Vivienda (sin sufijo)
    'p21', 'p22', 'p23', 'p24', 'p25', 'p26', 'p28', 'p31',
    'p32_1', 'p32_2', 'p32_9', 'p32_12',
    'tot_hog', 'pobviv', 'ur2'
]

# Verificar que todas existan antes de seleccionar
faltantes = [c for c in columnas_relevantes if c not in df_integrado.columns]
print("Faltantes:", faltantes)

df_proc = df_integrado[columnas_relevantes].copy()
print("Shape del subconjunto:", df_proc.shape)
df_proc.head()

Faltantes: []
Shape del subconjunto: (65862, 33)


,ent,con,upm,n_pro_viv,v_sel,per,tcti,tonp,tocup,tqdna,con5a17,escol_jefe,cs_p13_2,sex_jef,clase_hog,tam_hog,p_tot,clase2_jef,p21,p22,p23,p24,p25,p26,p28,p31,p32_1,p32_2,p32_9,p32_12,tot_hog,pobviv,ur2
0,01,40002,0100812,0032,01,422,0,0,0,0,0,0,0,0,0,0,0,0,,NaN,NaN,,,,,,,,,,1,0,1
1,01,40002,0100812,0033,02,422,0,0,0,0,1,4,3,1,2,2,4,1,3,02,04,1,1,1,1,1,1,1,2,1,1,4,1
2,01,40002,0100812,0039,03,422,0,0,0,0,2,4,3,2,2,2,4,4,1,03,05,1,1,1,4,1,1,1,1,1,1,4,1
3,01,40002,0100812,0056,04,422,0,0,0,0,0,0,0,0,0,0,0,0,3,01,06,1,1,1,3,1,1,1,1,1,1,0,1
4,01,40002,0100812,0079,05,422,0,0,0,0,1,2,3,2,3,2,4,4,2,03,06,1,1,1,1,1,1,1,1,1,1,4,1


In [11]:
mapa_columnas = {
    # Llaves
    'ent': 'entidad',
    'con': 'control',
    'upm': 'upm',
    'n_pro_viv': 'num_vivienda',
    'v_sel': 'vivienda_sel',
    'per': 'periodo',
    # Hogar
    'tcti': 'trabajo_infantil_total',
    'tonp': 'menores_ocupacion_no_permitida',
    'tocup': 'menores_ocupados_5a17',
    'tqdna': 'menores_quehaceres_no_adecuados',
    'con5a17': 'poblacion_5a17',
    'escol_jefe': 'escolaridad_jefe',
    'cs_p13_2': 'anios_aprobados_jefe',
    'sex_jef': 'sexo_jefe',
    'clase_hog': 'clase_hogar',
    'tam_hog': 'tamano_hogar',
    'p_tot': 'total_residentes',
    'clase2_jef': 'actividad_jefe',
    # Vivienda
    'p21': 'material_piso',
    'p22': 'cuartos_dormir',
    'p23': 'cuartos_totales',
    'p24': 'luz_electrica',
    'p25': 'agua_entubada',
    'p26': 'origen_agua',
    'p28': 'sanitario',
    'p31': 'drenaje',
    'p32_1': 'refrigerador',
    'p32_2': 'lavadora',
    'p32_9': 'computadora',
    'p32_12': 'internet',
    'tot_hog': 'hogares_en_vivienda',
    'pobviv': 'poblacion_vivienda',
    'ur2': 'ambito',
}

df_proc = df_proc.rename(columns=mapa_columnas)
print("Columnas finales:", df_proc.columns.tolist())
print("\nShape:", df_proc.shape)
df_proc.head()

Columnas finales: ['entidad', 'control', 'upm', 'num_vivienda', 'vivienda_sel', 'periodo', 'trabajo_infantil_total', 'menores_ocupacion_no_permitida', 'menores_ocupados_5a17', 'menores_quehaceres_no_adecuados', 'poblacion_5a17', 'escolaridad_jefe', 'anios_aprobados_jefe', 'sexo_jefe', 'clase_hogar', 'tamano_hogar', 'total_residentes', 'actividad_jefe', 'material_piso', 'cuartos_dormir', 'cuartos_totales', 'luz_electrica', 'agua_entubada', 'origen_agua', 'sanitario', 'drenaje', 'refrigerador', 'lavadora', 'computadora', 'internet', 'hogares_en_vivienda', 'poblacion_vivienda', 'ambito']

Shape: (65862, 33)


,entidad,control,upm,num_vivienda,vivienda_sel,periodo,trabajo_infantil_total,menores_ocupacion_no_permitida,menores_ocupados_5a17,menores_quehaceres_no_adecuados,poblacion_5a17,escolaridad_jefe,anios_aprobados_jefe,sexo_jefe,clase_hogar,tamano_hogar,total_residentes,actividad_jefe,material_piso,cuartos_dormir,cuartos_totales,luz_electrica,agua_entubada,origen_agua,sanitario,drenaje,refrigerador,lavadora,computadora,internet,hogares_en_vivienda,poblacion_vivienda,ambito
0,01,40002,0100812,0032,01,422,0,0,0,0,0,0,0,0,0,0,0,0,,NaN,NaN,,,,,,,,,,1,0,1
1,01,40002,0100812,0033,02,422,0,0,0,0,1,4,3,1,2,2,4,1,3,02,04,1,1,1,1,1,1,1,2,1,1,4,1
2,01,40002,0100812,0039,03,422,0,0,0,0,2,4,3,2,2,2,4,4,1,03,05,1,1,1,4,1,1,1,1,1,1,4,1
3,01,40002,0100812,0056,04,422,0,0,0,0,0,0,0,0,0,0,0,0,3,01,06,1,1,1,3,1,1,1,1,1,1,0,1
4,01,40002,0100812,0079,05,422,0,0,0,0,1,2,3,2,3,2,4,4,2,03,06,1,1,1,1,1,1,1,1,1,1,4,1


In [12]:
for col in ['entidad', 'control', 'upm', 'num_vivienda', 'vivienda_sel', 'periodo']:
    df_proc[col] = df_proc[col].astype(str)

In [13]:
# Cargar el CSV original de Vivienda con keep_default_na=False
df_viv_raw = pd.read_csv(
    'data/raw/conjunto_de_datos_viv_enti_2022.csv',
    dtype=str,
    encoding='utf-8',
    keep_default_na=False  # no convierte celdas vacías en NaN
)

# Ver cuántas filas tienen p21 vacío en el CSV original
filas_p21_vacio = (df_viv_raw['p21'].str.strip() == '').sum()
print(f"Filas con p21 vacío en el CSV original de Vivienda: {filas_p21_vacio}")

# Ver una muestra
df_viv_raw[df_viv_raw['p21'].str.strip() == ''][['ent', 'con', 'upm', 'n_pro_viv', 'p21', 'p24', 'p25']].head(10)

Filas con p21 vacío en el CSV original de Vivienda: 7618


,ent,con,upm,n_pro_viv,p21,p24,p25
0,01,40002,0100812,0032,,,
9,01,40006,0102027,0054,,,
23,01,40020,0100468,0125,,,
29,01,40022,0100071,0032,,,
33,01,40022,0100071,0123,,,
64,01,40029,0101841,0020,,,
73,01,40030,0101705,0117,,,
92,01,40039,0100043,0040,,,
95,01,40044,0101768,0011,,,
98,01,40044,0101768,0069,,,


In [14]:
# Eliminar filas con nulos en las variables clave de vivienda
vars_vivienda = ['material_piso', 'cuartos_dormir', 'cuartos_totales',
                 'luz_electrica', 'agua_entubada', 'sanitario', 'drenaje']

df_limpio = df_proc.dropna(subset=vars_vivienda).copy()
print("Antes:", df_proc.shape)
print("Después:", df_limpio.shape)
print("Filas eliminadas:", df_proc.shape[0] - df_limpio.shape[0])

Antes: (65862, 33)
Después: (58244, 33)
Filas eliminadas: 7618


In [15]:
df_limpio['origen_agua'] = df_limpio['origen_agua'].fillna(-1)
# Donde -1 significa "No aplica (sin agua entubada)"

In [18]:
print(df_proc['trabajo_infantil_total'].value_counts().sort_index())

trabajo_infantil_total
0    56341
1     7774
2     1457
3      258
4       25
5        7
Name: count, dtype: int64


In [19]:
# Forzar conversión a número
df_limpio['trabajo_infantil_total'] = pd.to_numeric(
    df_limpio['trabajo_infantil_total'], errors='coerce'
)

# Verificar tipo
print(df_limpio['trabajo_infantil_total'].dtype)

# Ahora sí, la comparación funciona
print("\nPorcentaje con TI > 0:",
      (df_limpio['trabajo_infantil_total'] > 0).mean() * 100)

int64

Porcentaje con TI > 0: 16.3467481629009


In [20]:
print(df_limpio['trabajo_infantil_total'].value_counts().sort_index())
print("\nPorcentaje con TI > 0:", (df_limpio['trabajo_infantil_total'] > 0).mean() * 100)

trabajo_infantil_total
0    48723
1     7774
2     1457
3      258
4       25
5        7
Name: count, dtype: int64

Porcentaje con TI > 0: 16.3467481629009


In [21]:
df_limpio['tiene_trabajo_infantil'] = (df_limpio['trabajo_infantil_total'] > 0).astype(int)

print("Distribución binaria:")
print(df_limpio['tiene_trabajo_infantil'].value_counts())
print("\nProporción:", df_limpio['tiene_trabajo_infantil'].mean().round(4))

Distribución binaria:
tiene_trabajo_infantil
0    48723
1     9521
Name: count, dtype: int64

Proporción: 0.1635


In [22]:
# Rellenar nulos de origen_agua con -1 = "No aplica"
df_limpio['origen_agua'] = df_limpio['origen_agua'].fillna(-1)

# Verificar
print(df_limpio['origen_agua'].value_counts().sort_index())

origen_agua
      1528
1    50997
2     3224
3     1306
4      516
5      134
6       37
7      328
9      174
Name: count, dtype: int64


In [23]:
mapa_entidades = {
    '01': 'Aguascalientes', '02': 'Baja California', '03': 'Baja California Sur',
    '04': 'Campeche', '05': 'Coahuila', '06': 'Colima', '07': 'Chiapas',
    '08': 'Chihuahua', '09': 'Ciudad de México', '10': 'Durango',
    '11': 'Guanajuato', '12': 'Guerrero', '13': 'Hidalgo', '14': 'Jalisco',
    '15': 'Estado de México', '16': 'Michoacán', '17': 'Morelos',
    '18': 'Nayarit', '19': 'Nuevo León', '20': 'Oaxaca', '21': 'Puebla',
    '22': 'Querétaro', '23': 'Quintana Roo', '24': 'San Luis Potosí',
    '25': 'Sinaloa', '26': 'Sonora', '27': 'Tabasco', '28': 'Tamaulipas',
    '29': 'Tlaxcala', '30': 'Veracruz', '31': 'Yucatán', '32': 'Zacatecas'
}

df_limpio['entidad_nombre'] = df_limpio['entidad'].map(mapa_entidades)
print(df_limpio['entidad_nombre'].value_counts().head())

entidad_nombre
Coahuila               2837
Baja California Sur    2279
Campeche               2164
Chihuahua              2123
Guanajuato             2049
Name: count, dtype: int64


In [24]:
df_limpio.to_csv('data/processed/inegi_procesado.csv', index=False)
print("Guardado:", df_limpio.shape)
print("Columnas:", df_limpio.columns.tolist())

Guardado: (58244, 35)
Columnas: ['entidad', 'control', 'upm', 'num_vivienda', 'vivienda_sel', 'periodo', 'trabajo_infantil_total', 'menores_ocupacion_no_permitida', 'menores_ocupados_5a17', 'menores_quehaceres_no_adecuados', 'poblacion_5a17', 'escolaridad_jefe', 'anios_aprobados_jefe', 'sexo_jefe', 'clase_hogar', 'tamano_hogar', 'total_residentes', 'actividad_jefe', 'material_piso', 'cuartos_dormir', 'cuartos_totales', 'luz_electrica', 'agua_entubada', 'origen_agua', 'sanitario', 'drenaje', 'refrigerador', 'lavadora', 'computadora', 'internet', 'hogares_en_vivienda', 'poblacion_vivienda', 'ambito', 'tiene_trabajo_infantil', 'entidad_nombre']
